In [1]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
from torchvision import transforms
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [2]:
class OxfordPetDataset(Dataset):
    def __init__(self, root, transform=None):
        self.root = root
        self.transform = transform

        self.files = [f for f in os.listdir(root) if f.lower().endswith(".jpg")]
        self.files.sort()

        # Extract class names
        class_names = [self._extract_class_name(f) for f in self.files]
        self.classes = sorted(list(set(class_names)))

        # Map class → index
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}

        # Build label list
        self.labels = [self.class_to_idx[self._extract_class_name(f)] for f in self.files]

    def _extract_class_name(self, filename):
        # Remove trailing "_<number>.jpg"
        base = filename.rsplit("_", 1)[0]
        return base

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.root, self.files[idx])
        img = Image.open(img_path).convert("RGB")
        label = self.labels[idx]

        if self.transform:
            img = self.transform(img)

        return img, label

In [3]:
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),

    transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
    ),
])

transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),

    transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
    ),
])

root = "../datasets/oxford-iiit-pet/images/images"

full_dataset = OxfordPetDataset(root, transform=transform_train)
num_classes = len(full_dataset.classes)
print("Classes:", num_classes)

Classes: 37


In [4]:
total = len(full_dataset)
train_size = int(0.7 * total)
val_size = int(0.15 * total)
test_size = total - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size]
)

# Validation/test use deterministic transforms
val_dataset.dataset.transform = transform_test
test_dataset.dataset.transform = transform_test

batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)

print("Train:", len(train_dataset), "Val:", len(val_dataset), "Test:", len(test_dataset))

Train: 5173 Val: 1108 Test: 1109


In [5]:
class ResidualBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_c)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_c != out_c:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_c, out_c, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_c)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out)


class ScalableResNetLite(nn.Module):
    def __init__(self, channels=64, depth=3, num_classes=num_classes):
        super().__init__()

        self.conv1 = nn.Conv2d(3, channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)

        layers = []
        c = channels
        for i in range(depth):
            stride = 1 if i == 0 else 2
            out_c = c if i == 0 else c * 2
            layers.append(ResidualBlock(c, out_c, stride))
            c = out_c

        self.res_layers = nn.Sequential(*layers)
        self.linear = nn.Linear(c, num_classes)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.res_layers(out)
        out = F.avg_pool2d(out, out.size()[3])
        out = out.view(out.size(0), -1)
        return self.linear(out)

In [6]:
def make_optimizer(opt_name, model, lr, wd):
    if opt_name == "sgd":
        return optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=wd)
    else:
        return optim.Adam(model.parameters(), lr=lr, weight_decay=wd)

In [7]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    correct, total, running_loss = 0, 0, 0.0

    # Wrap loader in tqdm for a live progress bar
    for x, y in tqdm(loader, desc="Training", leave=False):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    return correct / total, running_loss / total


def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for x, y in tqdm(loader, desc="Evaluating", leave=False):
            x, y = x.to(device), y.to(device)
            logits = model(x)
            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    return correct / total

In [8]:
# -----------------------------------------
# Use a smaller subset for sensitivity analysis
# -----------------------------------------
sa_subset_size = 1000  # you can lower to 500 if needed

sa_train_subset, _ = random_split(
    train_dataset,
    [sa_subset_size, len(train_dataset) - sa_subset_size]
)

train_loader_sa = DataLoader(
    sa_train_subset, batch_size=batch_size, shuffle=True,
    num_workers=0, pin_memory=True
)

In [9]:
#next(iter(train_loader_sa))

In [10]:
len(full_dataset.files)

7390

In [11]:
lrs = [1e-4, 3e-4, 1e-3]
channels_list = [48, 64, 96]
depth_list = [4, 5, 6]
opts = ["sgd", "adam"]
weight_decays = [0, 1e-4, 5e-4]

criterion = nn.CrossEntropyLoss()

coarse_results = []

for lr in lrs:
    for ch in channels_list:
        for d in depth_list:
            for opt in opts:
                for wd in weight_decays:
                    print("\nStarting config:", lr, ch, d, opt, wd)
                    model = ScalableResNetLite(channels=ch, depth=d).to(device)
                    optimizer = make_optimizer(opt, model, lr, wd)

                    print("Traiining for 1 epoch...")
                    train_acc, _ = train_one_epoch(model, train_loader_sa, criterion, optimizer)
                    print("Evaluating...")
                    val_acc = evaluate(model, val_loader)

                    coarse_results.append(
                        ({"lr": lr, "channels": ch, "depth": d, "opt": opt, "wd": wd},
                         val_acc)
                    )
                    print("SA:", lr, ch, d, opt, wd, "val_acc=", val_acc)

coarse_results.sort(key=lambda x: x[1], reverse=True)


Starting config: 0.0001 48 4 sgd 0
Traiining for 1 epoch...


Evaluating...


SA: 0.0001 48 4 sgd 0 val_acc= 0.024368231046931407

Starting config: 0.0001 48 4 sgd 0.0001
Traiining for 1 epoch...


Evaluating...


SA: 0.0001 48 4 sgd 0.0001 val_acc= 0.021660649819494584

Starting config: 0.0001 48 4 sgd 0.0005
Traiining for 1 epoch...


...

SA: 0.001 96 6 adam 0 val_acc= 0.016245487364620937

Starting config: 0.001 96 6 adam 0.0001
Traiining for 1 epoch...


Evaluating...


SA: 0.001 96 6 adam 0.0001 val_acc= 0.02888086642599278

Starting config: 0.001 96 6 adam 0.0005
Traiining for 1 epoch...


Evaluating...


SA: 0.001 96 6 adam 0.0005 val_acc= 0.030685920577617327


In [12]:
def derive_refined_space(coarse_results, top_frac=0.2):
    top_k = max(1, int(len(coarse_results) * top_frac))
    top_configs = [cfg for (cfg, acc) in coarse_results[:top_k]]

    lrs = [c["lr"] for c in top_configs]
    channels = [c["channels"] for c in top_configs]
    depths = [c["depth"] for c in top_configs]
    opts = [c["opt"] for c in top_configs]
    wds = [c["wd"] for c in top_configs]

    return {
        "lr": (min(lrs), max(lrs)),
        "channels": sorted(set(channels)),
        "depth": sorted(set(depths)),
        "opt": sorted(set(opts)),
        "wd": sorted(set(wds)),
    }

refined_space = derive_refined_space(coarse_results)
print("Refined space:", refined_space)

Refined space: {'lr': (0.0001, 0.001), 'channels': [48, 64, 96], 'depth': [4, 5, 6], 'opt': ['adam', 'sgd'], 'wd': [0, 0.0001, 0.0005]}


In [13]:
def sample_config(space):
    lr_min, lr_max = space["lr"]
    lr = 10 ** random.uniform(np.log10(lr_min), np.log10(lr_max))
    ch = random.choice(space["channels"])
    d = random.choice(space["depth"])
    opt = random.choice(space["opt"])
    wd = random.choice(space["wd"])
    return {"lr": lr, "channels": ch, "depth": d, "opt": opt, "wd": wd}

In [14]:
num_trials = 20
rand_results = []

for t in range(num_trials):
    config = sample_config(refined_space)
    model = ScalableResNetLite(config["channels"], config["depth"]).to(device)
    optimizer = make_optimizer(config["opt"], model, config["lr"], config["wd"])

    train_acc, _ = train_one_epoch(model, train_loader, criterion, optimizer)
    val_acc = evaluate(model, val_loader)

    rand_results.append((config, val_acc))
    print("RS trial", t, config, "val_acc=", val_acc)

rand_results.sort(key=lambda x: x[1], reverse=True)
best_config, best_val = rand_results[0]
print("Best config:", best_config, "val_acc=", best_val)

RS trial 0 {'lr': np.float64(0.0007714313103343947), 'channels': 48, 'depth': 4, 'opt': 'adam', 'wd': 0.0005} val_acc= 0.08212996389891697


RS trial 1 {'lr': np.float64(0.00037917293646104607), 'channels': 64, 'depth': 4, 'opt': 'adam', 'wd': 0.0005} val_acc= 0.0776173285198556


RS trial 2 {'lr': np.float64(0.0006828313547337804), 'channels': 64, 'depth': 4, 'opt': 'sgd', 'wd': 0} val_acc= 0.07581227436823104


RS trial 3 {'lr': np.float64(0.00026819665697183994), 'channels': 48, 'depth': 4, 'opt': 'adam', 'wd': 0.0005} val_acc= 0.08844765342960288


RS trial 4 {'lr': np.float64(0.0004326499710991801), 'channels': 64, 'depth': 4, 'opt': 'sgd', 'wd': 0} val_acc= 0.06859205776173286


...

RS trial 15 {'lr': np.float64(0.0003441561281562383), 'channels': 64, 'depth': 4, 'opt': 'sgd', 'wd': 0} val_acc= 0.0740072202166065


RS trial 16 {'lr': np.float64(0.00014983328078287155), 'channels': 96, 'depth': 5, 'opt': 'adam', 'wd': 0.0001} val_acc= 0.10379061371841156


RS trial 17 {'lr': np.float64(0.0009318626775737541), 'channels': 96, 'depth': 5, 'opt': 'adam', 'wd': 0} val_acc= 0.052346570397111915


RS trial 18 {'lr': np.float64(0.000973217377889648), 'channels': 64, 'depth': 6, 'opt': 'adam', 'wd': 0} val_acc= 0.05685920577617329


RS trial 19 {'lr': np.float64(0.000586333928674502), 'channels': 64, 'depth': 6, 'opt': 'adam', 'wd': 0} val_acc= 0.0703971119133574
Best config: {'lr': np.float64(0.00014983328078287155), 'channels': 96, 'depth': 5, 'opt': 'adam', 'wd': 0.0001} val_acc= 0.10379061371841156


In [18]:
final_model = ScalableResNetLite(best_config["channels"], best_config["depth"]).to(device)
optimizer = make_optimizer(best_config["opt"], final_model,
                           best_config["lr"], best_config["wd"])
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

best_val = 0.57
num_epochs = 50
for epoch in range(num_epochs):
    train_acc, train_loss = train_one_epoch(final_model, train_loader, criterion, optimizer)
    val_acc = evaluate(final_model, val_loader)
    if val_acc > best_val:
        best_val = val_acc
        torch.save(final_model.state_dict(), "../saved_models/pets_classifier.pth")
    print(epoch, "train_acc=", train_acc, "val_acc=", val_acc)

    scheduler.step()

final_model.load_state_dict(torch.load("../saved_models/pets_classifier.pth"))
test_acc = evaluate(final_model, test_loader)
print("Final test accuracy:", test_acc)

0 train_acc= 0.06649913009858882 val_acc= 0.10018050541516245


1 train_acc= 0.11463367485018365 val_acc= 0.11101083032490974


2 train_acc= 0.16547457954765127 val_acc= 0.19945848375451264


3 train_acc= 0.2244345640827373 val_acc= 0.11823104693140794


4 train_acc= 0.2627102261743669 val_acc= 0.19584837545126355


...

46 train_acc= 1.0 val_acc= 0.6290613718411552


47 train_acc= 1.0 val_acc= 0.6425992779783394


48 train_acc= 1.0 val_acc= 0.6407942238267148


49 train_acc= 1.0 val_acc= 0.6416967509025271


Final test accuracy: 0.6104598737601443
